In [6]:
!pip install pymupdf



     --------------------------------------- 18.7/18.7 MB 16.8 MB/s eta 0:00:00


In [13]:
# Filename: extractor.py (for Round 1A)
import fitz  # PyMuPDF
import os
import json
from collections import defaultdict

def extract_headings_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    outline = []
    font_stats = defaultdict(int)

    # Pass 1: Collect font sizes
    for page in doc:
        blocks = page.get_text("dict")['blocks']
        for b in blocks:
            for line in b.get("lines", []):
                for span in line.get("spans", []):
                    font_stats[round(span["size"], 1)] += 1

    # Determine thresholds
    sizes = sorted(font_stats.items(), key=lambda x: -x[0])
    size_levels = {size[0]: idx for idx, size in enumerate(sizes[:4])}  # Top 4 sizes
    
    # Pass 2: Extract headings
    for page_number, page in enumerate(doc, start=1):
        blocks = page.get_text("dict")['blocks']
        for b in blocks:
            for line in b.get("lines", []):
                for span in line.get("spans", []):
                    size = round(span["size"], 1)
                    text = span["text"].strip()
                    if len(text) < 3 or not text[0].isalpha():
                        continue

                    level = size_levels.get(size, None)
                    if level is not None and level < 3:
                        outline.append({
                            "level": f"H{level+1}",
                            "text": text,
                            "page": page_number
                        })

    # Guess title from largest heading on page 1
    title = next((h['text'] for h in outline if h['page'] == 1 and h['level'] == 'H1'), 'Untitled')
    return {
        "title": title,
        "outline": outline
    }

def process_directory(input_dir, output_dir):
    for filename in os.listdir(input_dir):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename.replace(".pdf", ".json"))
            try:
                result = extract_headings_from_pdf(pdf_path)
                with open(output_path, "w") as f:
                    json.dump(result, f, indent=2)
                print(f"Processed: {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {e}")

if __name__ == "__main__":
    process_directory("input", "output")  # Use local paths


Processed: sample.pdf
